### update HAR GENES LIST

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

/scratch200/reutj/conda-envs/jupyter-scanpy/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/scratch200/reutj/conda-envs/jupyter-scanpy/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/scratch200/reutj/conda-envs/jupyter-scanpy/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/scratch200/reutj/conda-envs/jupyter-scanpy/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/scratch200/reutj/conda-envs/jupyter-scanpy/lib/python3.12/site-packag

In [2]:
har_genes=pd.read_excel("/scratch200/reutj/data/hsg_gene_lists/har_associated_genes.xlsx",header=1)

In [180]:
har_genes

,HAR-associated genes,HGE:FB-associated genes,HGE:AB-associated genes,HLE-associated genes,Unnamed: 4,HAR-associated genes.1,HGE:FB-associated genes.1,HGE:AB-associated genes.1,HLE-associated genes.1
0,ENSG00000002822,ENSG00000002822,ENSG00000001460,ENSG00000004897,NaN,MAD1L1,MAD1L1,STPG1,CDC27
1,ENSG00000004897,ENSG00000005020,ENSG00000001626,ENSG00000005059,NaN,CDC27,SKAP2,CFTR,CCDC109B
2,ENSG00000005020,ENSG00000013561,ENSG00000003400,ENSG00000006327,NaN,SKAP2,RNF14,CASP10,TNFRSF12A
3,ENSG00000005955,ENSG00000013810,ENSG00000005471,ENSG00000006607,NaN,GGNBP2,TACC3,ABCB4,FARP2
4,ENSG00000009724,ENSG00000014164,ENSG00000005486,ENSG00000006715,NaN,MASP2,ZC3H3,RHBDD2,VPS41
...,...,...,...,...,...,...,...,...,...
1678,ENSG00000272407,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1679,ENSG00000272463,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1680,ENSG00000272692,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1681,ENSG00000272862,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
missing_symb_list=har_genes.iloc[1078:,0].tolist()

In [189]:
len(missing_symb_list)

605

In [6]:
from gseapy import Biomart
# Fetch mapping from Ensembl Biomart
bm = Biomart(host="useast.ensembl.org")
queries ={'ensembl_gene_id': missing_symb_list[:100]} # need to be a dict object
results = bm.query(dataset='hsapiens_gene_ensembl',
                   attributes=['ensembl_gene_id', 'external_gene_name'],
                   filters=queries)

2025-01-16 15:15:38,510 [WARNING] host useast.ensembl.org is not reachable, will try useast.ensembl.org 


ParseError: mismatched tag: line 62, column 2 (<string>)

In [9]:
!pip install mygene

In [14]:
import mygene

mg = mygene.MyGeneInfo()

# Query MyGene.info for gene symbols
results = mg.querymany(missing_symb_list, scopes="ensembl.gene", fields="symbol", species="human")

# Convert results to a dictionary
ensg_to_symbol = {item["query"]: item.get("symbol", None) for item in results}

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
72 input query terms found no hit:	['ENSG00000108272', 'ENSG00000129282', 'ENSG00000174083', 'ENSG00000182230', 'ENSG00000197445', 'ENS


In [22]:
new_har_symb=pd.DataFrame({'ens':ensg_to_symbol.keys(),'gene':ensg_to_symbol.values()})
new_har_wsymb=new_har_symb.loc[[str(i)!='None' for i in new_har_symb.gene],:]
new_har_wosymb=new_har_symb.loc[[str(i)=='None' for i in new_har_symb.gene],:]

In [25]:
har_genes=har_genes.rename(columns={'HAR-associated genes':'ens','HAR-associated genes.1':'gene'})

In [36]:
har_genes_wsymb=pd.concat([har_genes.iloc[:1078,[0,5]],new_har_wsymb]).reset_index()
full_har_list=pd.concat([har_genes_wsymb,new_har_wosymb]).reset_index()
full_har_list=full_har_list.drop(columns=['level_0','index'])
full_har_list

,ens,gene
0,ENSG00000002822,MAD1L1
1,ENSG00000004897,CDC27
2,ENSG00000005020,SKAP2
3,ENSG00000005955,GGNBP2
4,ENSG00000009724,MASP2
...,...,...
1678,ENSG00000272407,None
1679,ENSG00000272463,None
1680,ENSG00000272692,None
1681,ENSG00000272862,None


In [37]:
#save to file
full_har_list.to_csv("/scratch200/reutj/data/hsg_gene_lists/updated_har_list.csv")

In [46]:
har_gene_symb=full_har_list.iloc[:1511,1].tolist()

In [47]:
#update custom_har_genes gmt list
output_file="/scratch200/reutj/data/hsg_gene_lists/custom_har_genes_list.gmt"
with open(output_file, 'w') as f:
    # Create a line in GMT format for one gene set
    line = f"{har_genes}\tDescription\t{'\t'.join(har_gene_symb)}\n"
    f.write(line)